In [1]:
pip install h5netcdf h5py

   ---------------------------------------- 0.0/65.3 kB ? eta -:--:--
   ------------------------- -------------- 41.0/65.3 kB 991.0 kB/s eta 0:00:01
   ---------------------------------------- 65.3/65.3 kB 1.8 MB/s eta 0:00:00
Note: you may need to restart the kernel to use updated packages.


In [3]:
"""Précalcule, pour chaque maille de 0,25°, les grandeurs de l'onglet Grêle de CatMap : moyenne annuelle d'événements ≥ 5 cm, tendance par décennie,
p-value et niveaux annuels de périodes de retour 2, 5, 10, 20 et 50 ans. Produit UN fichier ``hail_compact.npz`` (à déposer ensuite dans data/grele/ du dépôt).

Script autonome : il n'a besoin que de numpy et d'une bibliothèque NetCDF (netCDF4 ou h5netcdf, déjà présente si vous utilisez xarray). Il peut être lancé
depuis n'importe quel dossier.

Usage :
    python build_hail_compact.py                      -> lit le dossier où se trouve le script, écrit hail_compact.npz dans ce même dossier
    python build_hail_compact.py DOSSIER_SOURCE [SORTIE.npz]

Dossier source : prob_hail_ge5cm_1950.nc … prob_hail_ge5cm_2023.nc et hail_ge5cm_trends_1950_2023.nc (hail_ge5cm_climatology_1950_2023.nc facultatif : contrôle).
Mémoire nécessaire : environ 1 Go. Licence des données (CC BY-NC-ND 4.0) : le fichier produit est dérivé ; ne le publiez pas si votre dépôt est public.
"""
from __future__ import annotations

import math
import re
import sys
from pathlib import Path
from typing import Optional

import numpy as np

YEARS = tuple(range(1950, 2024))
RPS = (2, 5, 10, 20, 50)
TREND_FILE = "hail_ge5cm_trends_1950_2023.nc"
CLIM_FILE = "hail_ge5cm_climatology_1950_2023.nc"
LAYER_PREFIX = "rp"


class HailError(Exception):
    pass


def annual_file(year):
    return f"prob_hail_ge5cm_{int(year)}.nc"


def _fill(arr: np.ndarray, attrs: dict) -> np.ndarray:
    a = np.asarray(arr)
    if a.dtype.kind not in "fiu":
        return a
    a = a.astype(float)
    for k in ("_FillValue", "missing_value"):
        v = attrs.get(k)
        if v is not None:
            for fv in np.atleast_1d(v):
                try:
                    a = np.where(a == float(fv), np.nan, a)
                except (TypeError, ValueError):
                    pass
    sc, off = attrs.get("scale_factor"), attrs.get("add_offset")
    if sc is not None:
        a = a * float(np.atleast_1d(sc)[0])
    if off is not None:
        a = a + float(np.atleast_1d(off)[0])
    return a


def _txt(v):
    return v.decode("utf-8", "replace") if isinstance(v, bytes) else v


def read_dataset(path) -> dict:
    """Lit un fichier NetCDF (NetCDF4/HDF5 ou NetCDF3) : ``{"attrs", "vars": {nom: {"dims", "data", "attrs"}}}``. Essaie netCDF4, h5netcdf, scipy."""
    path = str(path)
    errors = []
    try:
        import netCDF4
        ds = netCDF4.Dataset(path)
        try:
            vs = {}
            for n, v in ds.variables.items():
                at = {k: v.getncattr(k) for k in v.ncattrs()}
                vs[n] = {"dims": tuple(v.dimensions), "data": np.ma.filled(np.ma.asarray(v[:]).astype(float), np.nan)
                         if v.dtype.kind in "fiu" else np.asarray(v[:]), "attrs": at}
            return {"attrs": {k: ds.getncattr(k) for k in ds.ncattrs()}, "vars": vs}
        finally:
            ds.close()
    except ImportError:
        errors.append("netCDF4 absent")
    except Exception as e:                                      # noqa: BLE001
        errors.append(f"netCDF4 : {type(e).__name__} {str(e)[:80]}")
    try:
        import h5netcdf
        with h5netcdf.File(path, "r") as ds:
            vs = {}
            for n, v in ds.variables.items():
                at = {k: v.attrs[k] for k in v.attrs}
                vs[n] = {"dims": tuple(v.dimensions), "data": _fill(v[...], at), "attrs": at}
            return {"attrs": {k: ds.attrs[k] for k in ds.attrs}, "vars": vs}
    except ImportError:
        errors.append("h5netcdf absent")
    except Exception as e:                                      # noqa: BLE001
        errors.append(f"h5netcdf : {type(e).__name__} {str(e)[:80]}")
    try:
        import xarray as xr
        ds = xr.open_dataset(path)
        try:
            vs = {n: {"dims": tuple(v.dims), "data": np.asarray(v.values), "attrs": dict(v.attrs)} for n, v in ds.variables.items()}
            return {"attrs": dict(ds.attrs), "vars": vs}
        finally:
            ds.close()
    except ImportError:
        errors.append("xarray absent")
    except Exception as e:                                      # noqa: BLE001
        errors.append(f"xarray : {type(e).__name__} {str(e)[:80]}")
    try:
        from scipy.io import netcdf_file
        with netcdf_file(path, "r", mmap=False) as ds:
            vs = {}
            for n, v in ds.variables.items():
                at = {k: _txt(getattr(v, k)) for k in getattr(v, "_attributes", {})}
                vs[n] = {"dims": tuple(v.dimensions), "data": _fill(np.array(v[:]), at), "attrs": at}
            return {"attrs": {k: _txt(getattr(ds, k)) for k in getattr(ds, "_attributes", {})}, "vars": vs}
    except Exception as e:                                      # noqa: BLE001
        errors.append(f"scipy : {type(e).__name__} {str(e)[:80]}")
    raise HailError("lecture NetCDF impossible (" + " ; ".join(errors) + "). Installez une bibliothèque NetCDF : pip install h5netcdf h5py (ou netCDF4).")


def _axis(ds: dict, names: tuple) -> Optional[tuple]:
    for n in names:
        for k, v in ds["vars"].items():
            if k.lower() == n and v["data"].ndim == 1:
                return k, np.asarray(v["data"], float)
    return None


def field(ds: dict, prefer: tuple = (), contains: tuple = ()) -> dict:
    """Champ 2-D (lat, lon) d'un jeu lu : ``{"name", "lat", "lon", "z", "attrs"}``. Variable choisie par nom préféré, sinon par mot contenu, sinon la seule candidate."""
    la, lo = _axis(ds, ("lat", "latitude", "y")), _axis(ds, ("lon", "longitude", "x"))
    if la is None or lo is None:
        raise HailError("axes lat/lon introuvables ; variables : " + ", ".join(ds["vars"]))
    cands = {k: v for k, v in ds["vars"].items() if v["data"].ndim >= 2 and la[0] in v["dims"] and lo[0] in v["dims"]}
    if not cands:
        raise HailError("aucune variable (lat, lon) ; variables : " + ", ".join(f"{k}{v['dims']}" for k, v in ds["vars"].items()))
    name = next((p for p in prefer if p in cands), None) or next((k for k in cands if any(c in k.lower() for c in contains)), None) or next(iter(cands))
    v = cands[name]
    z = np.asarray(v["data"], float)
    dims = list(v["dims"])
    keep = [i for i, d in enumerate(dims) if d in (la[0], lo[0])]
    for i in sorted((i for i in range(len(dims)) if i not in keep), reverse=True):
        if z.shape[i] != 1:
            raise HailError(f"variable {name} : dimension supplémentaire {dims[i]} de taille {z.shape[i]} (attendu : 1) ; dimensions {tuple(dims)}")
        z = np.take(z, 0, axis=i)
        dims.pop(i)
    if dims.index(la[0]) > dims.index(lo[0]):
        z = z.T
    return {"name": name, "lat": la[1], "lon": lo[1], "z": z, "attrs": v["attrs"]}


def rp_layers(A: np.ndarray, rps) -> dict:
    """Niveau annuel dépassé une année sur ``RP`` pour chaque maille : même quantile que ``rp_levels`` (positions de Weibull i/(n+1)), calculé sur toute la
    grille. ``A`` : (années, lat, lon). Mailles sans donnée : NaN ; RP > n + 1 : NaN."""
    n = A.shape[0]
    S = np.sort(A, axis=0)                                       # NaN en fin de tri
    nan = ~np.isfinite(A).all(axis=0)
    out = {}
    for rp in rps:
        if rp > n + 1 or n < 3:
            out[rp] = np.full(A.shape[1:], np.nan, np.float32)
            continue
        h = (1.0 - 1.0 / rp) * (n + 1.0)                          # rang (1-indexé) du quantile
        lo = min(max(int(math.floor(h)) - 1, 0), n - 1)
        hi = min(lo + 1, n - 1)
        frac = h - math.floor(h) if h >= 1 else 0.0
        v = S[lo] + frac * (S[hi] - S[lo])
        out[rp] = np.where(nan, np.nan, v).astype(np.float32)
    return out


def scan(src: Path, progress=print) -> dict:
    """Repère dans le dossier les seuls fichiers utiles (les autres .nc, par exemple ceux des villes, sont ignorés) :
    annuels ``prob_hail_ge5cm_<année>``, tendances (nom contenant « trend ») et climatologie (nom contenant « climatolog »)."""
    nc = sorted(p for p in Path(src).iterdir() if p.is_file() and p.suffix.lower() == ".nc")
    annual, trend, clim, ignored = {}, [], [], []
    for p in nc:
        n = p.name.lower()
        m = re.search(r"prob_hail_ge5cm[_\- ]*(\d{4})", n)
        if m and int(m.group(1)) in YEARS:
            y = int(m.group(1))
            if y not in annual or len(p.name) < len(annual[y].name):       # doublon du type « fichier (1).nc » : on garde le nom le plus court
                annual[y] = p
        elif "trend" in n and "hail" in n:
            trend.append(p)
        elif "climatolog" in n and "hail" in n:
            clim.append(p)
        else:
            ignored.append(p.name)
    progress(f"{len(nc)} fichier(s) .nc dans le dossier : {len(annual)} annuel(s), {len(trend)} de tendances, {len(clim)} de climatologie, {len(ignored)} ignoré(s)")
    if ignored:
        progress("  ignorés : " + ", ".join(ignored[:12]) + (" …" if len(ignored) > 12 else ""))
    miss = [y for y in YEARS if y not in annual]
    if miss and annual:
        progress(f"  années absentes ({len(miss)}) : " + ", ".join(map(str, miss[:10])) + (" …" if len(miss) > 10 else ""))
    return {"annual": annual, "trend": sorted(trend, key=lambda p: len(p.name))[:1], "clim": sorted(clim, key=lambda p: len(p.name))[:1]}


def build(src, out_path, rps=RPS, max_mb=90.0, progress=print):
    found = scan(src, progress)
    years = sorted(found["annual"])
    if len(years) < 10:
        raise HailError(f"fichiers annuels introuvables ou trop peu nombreux dans {src} ({len(years)} trouvé(s) ; attendu : prob_hail_ge5cm_AAAA.nc)")
    if len(years) < len(YEARS):
        progress(f"ATTENTION : {len(years)} année(s) sur {len(YEARS)} seulement ; les périodes de retour supérieures à {len(years) + 1} ans seront vides (NaN).")
    if not found["trend"]:
        raise HailError("fichier de tendances introuvable (nom attendu : hail_ge5cm_trends_1950_2023.nc)")
    arrs, ref = [], None
    for y in years:
        f = field(read_dataset(found["annual"][y]), contains=("hail", "prob"))
        ref = ref or f
        if f["z"].shape != ref["z"].shape:
            raise HailError(f"grille différente en {y}")
        arrs.append(f["z"].astype(np.float32))
        if len(arrs) % 10 == 0:
            progress(f"  {len(arrs)}/{len(years)} lus")
    A = np.stack(arrs)
    del arrs
    mean = np.nanmean(A, axis=0).astype(np.float32)
    layers = rp_layers(A, rps)
    del A
    tds = read_dataset(found["trend"][0])
    tr = field(tds, prefer=("trend_hail_ge5cm",), contains=("trend",))
    pv = field(tds, prefer=("pvalue_hail_ge5cm",), contains=("pvalue", "p_value"))
    if tr["z"].shape != ref["z"].shape:
        raise HailError("la grille du fichier de tendances diffère de celle des fichiers annuels.")
    if found["clim"]:
        cfile = field(read_dataset(found["clim"][0]), prefer=("hail_ge5cm",), contains=("hail",))["z"]
        ok = np.isfinite(cfile) & (mean > 1e-6)
        progress(f"contrôle : climatologie du fichier ÷ moyenne des années = {float(np.median(cfile[ok] / mean[ok])):.1f} "
                 f"(≈ {len(years)} si la climatologie est un cumul sur la période)")
    f16 = lambda a: np.asarray(a, np.float32).astype(np.float16)  # noqa: E731
    data = {"lat": ref["lat"].astype(np.float32), "lon": ref["lon"].astype(np.float32), "years": np.array(years, np.int16), "rps": np.array(list(rps), np.int16),
            "clim": f16(mean), "trend": f16(tr["z"]), "pvalue": f16(pv["z"])}
    data.update({f"{LAYER_PREFIX}{r}": f16(v) for r, v in layers.items()})
    np.savez_compressed(out_path, **data)
    mb = Path(out_path).stat().st_size / 1e6
    progress(f"écrit : {out_path} ({mb:.1f} Mo)")
    if mb > max_mb:
        raise HailError(f"fichier trop gros ({mb:.0f} Mo > {max_mb:g} Mo) : utilisez Git LFS ou recadrez les données.")
    return {"annees": len(years), "rps": list(rps), "taille_mo": round(mb, 2), "mailles": int(np.isfinite(mean).sum())}


def main(argv=None):
    """Ligne de commande ou notebook (Jupyter / Spyder) : sans argument, travaille dans le dossier du script, ou dans le dossier courant du notebook."""
    argv = [a for a in (sys.argv[1:] if argv is None else argv) if not a.startswith("-") and not a.endswith(".json")]
    try:
        here = Path(__file__).resolve().parent
    except NameError:                                           # notebook : pas de __file__
        here = Path.cwd()
    src = Path(argv[0]) if argv else here
    out = Path(argv[1]) if len(argv) > 1 else src / "hail_compact.npz"
    if not src.is_dir():
        print(f"Dossier source introuvable : {src}")
        return None
    print(f"Source : {src}\nSortie : {out}")
    try:
        res = build(src, out)
    except HailError as e:
        print(f"ERREUR : {e}")
        return None
    print(res)
    return res


if __name__ == "__main__":
    main()

Source : C:\Users\maxime.trevisani_squ\Desktop\Hail_data
Sortie : C:\Users\maxime.trevisani_squ\Desktop\Hail_data\hail_compact.npz
89 fichier(s) .nc dans le dossier : 74 annuel(s), 1 de tendances, 1 de climatologie, 13 ignoré(s)
  ignorés : correlation_hail_ge5cm_t2m.nc, hail_ge5cm_seasonal_Beijing.nc, hail_ge5cm_seasonal_Brisbane.nc, hail_ge5cm_seasonal_Dallas.nc, hail_ge5cm_seasonal_Johannesburg.nc, hail_ge5cm_seasonal_Mendoza.nc, hail_ge5cm_seasonal_Milan.nc, hail_ge5cm_stripes_Beijing.nc, hail_ge5cm_stripes_Brisbane.nc, hail_ge5cm_stripes_Dallas.nc, hail_ge5cm_stripes_Johannesburg.nc, hail_ge5cm_stripes_Mendoza.nc …
  10/74 lus
  20/74 lus
  30/74 lus
  40/74 lus
  50/74 lus
  60/74 lus
  70/74 lus
contrôle : climatologie du fichier ÷ moyenne des années = 74.0 (≈ 74 si la climatologie est un cumul sur la période)
écrit : C:\Users\maxime.trevisani_squ\Desktop\Hail_data\hail_compact.npz (4.9 Mo)
{'annees': 74, 'rps': [2, 5, 10, 20, 50], 'taille_mo': 4.9, 'mailles': 813600}
